# Gold layer — star schema

Builds the consumption-ready star schema over the silver layer:

| Table | Type | Grain |
|---|---|---|
| `dim_date` | dimension | one row per calendar day |
| `dim_store` | dimension | one row per store |
| `dim_product` | dimension | one row per product (EAN) |
| `fact_sales` | fact | store × product × day — units sold and returned, gross/net amounts |
| `fact_store_stock` | fact | store × product on the as-of date — stock position with in-transit, warehouse availability, depth target and flags |
| `fact_warehouse_stock` | fact | day × warehouse × product — units available |
| `fact_replenishment_decisions` | fact | store × product for the run date — the engine decision, units needed and ordered |

Facts carry surrogate keys (`date_key`, `store_key`, `product_key`) plus the natural keys, so both Genie and the "why" agent can join easily. Every table states its grain in its comment, per the brief.

In [0]:
import re

from pyspark.sql import functions as F


def _param(name, default):
    """Widget value if defined, otherwise the default, so the notebook also runs headless."""
    try:
        value = dbutils.widgets.get(name).strip()
        return value or default
    except Exception:
        return default


CATALOG = _param("catalog", "genie_agents_poc")
BRONZE = _param("bronze_schema", "bronze")
SILVER = _param("silver_schema", "silver")
GOLD = _param("gold_schema", "gold")

_IDENT = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
for label, value in [("catalog", CATALOG), ("bronze_schema", BRONZE),
                     ("silver_schema", SILVER), ("gold_schema", GOLD)]:
    if not _IDENT.match(value):
        raise ValueError(f"{label} may contain only letters, digits and underscores; got {value!r}")


def silver(table):
    return spark.table(f"{CATALOG}.{SILVER}.{table}")


def save_gold(df, table, grain):
    target = f"{CATALOG}.{GOLD}.{table}"
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)
    spark.sql(f"COMMENT ON TABLE {target} IS '{grain}'")
    print(f"  {table:<30}{spark.table(target).count():>10,} rows -> {target}")

In [0]:
from pyspark.sql import Window

# dim_date spans every date used anywhere: sales window through the as-of (run) date.
lo = silver("pos_sales").select(F.min("bill_date").alias("v")).first()["v"]
hi = silver("replenishment_decisions").select(F.max("run_date").alias("v")).first()["v"]

dim_date = (
    spark.range((hi - lo).days + 1)
    .select(F.date_add(F.lit(lo), F.col("id").cast("int")).alias("full_date"))
    .select(
        F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
        "full_date",
        F.year("full_date").alias("year"),
        F.quarter("full_date").alias("quarter"),
        F.month("full_date").alias("month"),
        F.date_format("full_date", "MMMM").alias("month_name"),
        F.dayofmonth("full_date").alias("day_of_month"),
        F.dayofweek("full_date").alias("day_of_week"),
        F.date_format("full_date", "EEEE").alias("day_name"),
        F.weekofyear("full_date").alias("week_of_year"),
        F.date_sub("full_date", F.dayofweek("full_date") - 1).alias("week_start_date"),
        F.dayofweek("full_date").isin(1, 7).alias("is_weekend"),
    )
)
save_gold(dim_date, "dim_date",
          "Gold date dimension - one row per calendar day covering the sales window and the as-of date.")

dim_store = (
    silver("stores")
    .withColumn("store_key", F.row_number().over(Window.orderBy("store_code")))
    .select("store_key", "store_code", "store_name", "city", "state", "region",
            "grade", "store_format", "status", "serving_warehouse")
)
save_gold(dim_store, "dim_store",
          "Gold store dimension - one row per store, snapshot (SCD type 1).")

dim_product = (
    silver("products")
    .withColumn("product_key", F.row_number().over(Window.orderBy("ean")))
    .select("product_key", "ean", "style_code", "style_name", "category", "colour",
            "size", "size_class", "is_pivotal", "mrp", "season")
)
save_gold(dim_product, "dim_product",
          "Gold product dimension - one row per product (EAN = style x colour x size).")

  dim_date                              90 rows -> genie_agents_poc.gold.dim_date


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


  dim_store                             40 rows -> genie_agents_poc.gold.dim_store
  dim_product                          440 rows -> genie_agents_poc.gold.dim_product


In [0]:
measure_cols = [
    F.coalesce(F.sum(F.when(F.col("qty") > 0, F.col("qty"))), F.lit(0)).cast("int").alias("units_sold"),
    F.coalesce(F.sum(F.when(F.col("qty") < 0, -F.col("qty"))), F.lit(0)).cast("int").alias("units_returned"),
    F.coalesce(F.sum(F.when(F.col("qty") > 0, F.col("qty") * F.col("mrp"))), F.lit(0.0)).alias("gross_amount"),
    F.coalesce(F.sum(F.when(F.col("qty") > 0,
                              F.col("qty") * F.col("mrp") * F.col("discount_pct") / 100.0)),
               F.lit(0.0)).alias("discount_amount"),
    F.coalesce(F.sum("net_amount"), F.lit(0.0)).alias("net_amount"),
    F.count(F.lit(1)).alias("txn_line_count"),
]

fact_sales = (
    silver("pos_sales")
    .groupBy("bill_date", "store_code", "ean")
    .agg(*measure_cols)
    .join(dim_date.select("date_key", F.col("full_date").alias("bill_date")), "bill_date")
    .join(dim_store.select("store_key", "store_code"), "store_code")
    .join(dim_product.select("product_key", "ean"), "ean")
    .select("date_key", "store_key", "product_key", "bill_date", "store_code", "ean",
            "units_sold", "units_returned", "gross_amount", "discount_amount",
            "net_amount", "txn_line_count")
)
save_gold(fact_sales, "fact_sales",
          "Gold fact_sales - one row per store x product x day. Measures: units_sold, "
          "units_returned, gross_amount, discount_amount, net_amount, txn_line_count. "
          "Returns are included as negative contributions to net_amount.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


  fact_sales                       135,067 rows -> genie_agents_poc.gold.fact_sales


In [0]:
# Stock position at the as-of date: store stock enriched with in-transit, warehouse
# availability, depth target, capacity and hold/clearance flags.
asof = silver("store_stock").select(F.max("snapshot_date").alias("d")).first()["d"]
wh_asof = (
    silver("warehouse_stock_daily").filter(F.col("snapshot_date") == F.lit(asof))
    .select(F.col("warehouse").alias("serving_warehouse"), "ean",
            F.col("available_qty").alias("wh_available"))
)
transit_qty = silver("in_transit").groupBy("store_code", "ean").agg(F.sum("qty").alias("in_transit_qty"))
holds = silver("style_holds").select("style_code", "hold_reason", F.lit(True).alias("is_on_hold"))
clearance = silver("clearance_styles").select("style_code", F.lit(True).alias("is_clearance"))

fact_store_stock = (
    silver("store_stock").select("snapshot_date", "store_code", "ean", "stock_on_hand")
    .join(dim_store.select("store_key", "store_code", "grade", "serving_warehouse"), "store_code")
    .join(dim_product.select("product_key", "ean", "style_code", "category", "size_class",
                            "is_pivotal"), "ean")
    .join(silver("depth_targets"), ["grade", "size_class"])
    .join(silver("store_capacity"), ["store_code", "category"])
    .join(transit_qty, ["store_code", "ean"], "left")
    .join(wh_asof, ["serving_warehouse", "ean"], "left")
    .join(holds, "style_code", "left")
    .join(clearance, "style_code", "left")
    .join(dim_date.select("date_key", F.col("full_date").alias("snapshot_date")), "snapshot_date")
    .fillna({"in_transit_qty": 0, "wh_available": 0})
    .withColumn("is_on_hold", F.coalesce("is_on_hold", F.lit(False)))
    .withColumn("is_clearance", F.coalesce("is_clearance", F.lit(False)))
    .withColumn("units_needed", F.greatest(
        F.lit(0), F.col("depth_target") - F.col("stock_on_hand") - F.col("in_transit_qty")))
    .select("date_key", "store_key", "product_key", "snapshot_date", "store_code", "ean",
            "stock_on_hand", "in_transit_qty", "wh_available", "depth_target", "units_needed",
            "max_units", "is_pivotal", "is_on_hold", "hold_reason", "is_clearance")
)
save_gold(fact_store_stock, "fact_store_stock",
          "Gold fact_store_stock - one row per store x product on the as-of snapshot date. "
          "Measures: stock_on_hand, in_transit_qty, wh_available, depth_target, units_needed, "
          "max_units. Flags: is_pivotal, is_on_hold, is_clearance.")

fact_warehouse_stock = (
    silver("warehouse_stock_daily")
    .join(dim_date.select("date_key", F.col("full_date").alias("snapshot_date")), "snapshot_date")
    .join(dim_product.select("product_key", "ean"), "ean")
    .select("date_key", "product_key", "snapshot_date", "warehouse", "ean", "available_qty")
)
save_gold(fact_warehouse_stock, "fact_warehouse_stock",
          "Gold fact_warehouse_stock - one row per day x warehouse x product; warehouse is a "
          "degenerate dimension.")

fact_replenishment_decisions = (
    silver("replenishment_decisions")
    .join(dim_date.select("date_key", F.col("full_date").alias("run_date")), "run_date")
    .join(dim_store.select("store_key", "store_code"), "store_code")
    .join(dim_product.select("product_key", "ean"), "ean")
    .select("date_key", "store_key", "product_key", "run_date", "store_code", "ean",
            "decision", "units_needed", "order_qty", "source_location")
)
save_gold(fact_replenishment_decisions, "fact_replenishment_decisions",
          "Gold fact_replenishment_decisions - one row per store x product for the run date; "
          "decision and quantities from the engine, source_location names the warehouse or "
          "donor store.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


  fact_store_stock                  11,199 rows -> genie_agents_poc.gold.fact_store_stock
  fact_warehouse_stock              79,200 rows -> genie_agents_poc.gold.fact_warehouse_stock
  fact_replenishment_decisions      11,199 rows -> genie_agents_poc.gold.fact_replenishment_decisions


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# Integrity checks: gold must reconcile with silver.
fs = spark.table(f"{CATALOG}.{GOLD}.fact_sales")
sp = silver("pos_sales")

silver_rows, gold_rows = sp.count(), fs.count()
silver_net = sp.agg(F.sum("net_amount").alias("v")).first()["v"]
gold_net = fs.agg(F.sum("net_amount").alias("v")).first()["v"]
silver_sold = sp.filter("qty > 0").agg(F.sum("qty").alias("v")).first()["v"]
gold_sold = fs.agg(F.sum("units_sold").alias("v")).first()["v"]

assert gold_rows == sp.select("bill_date", "store_code", "ean").distinct().count(), \
    "fact_sales rows != distinct store x product x day in silver"
assert round(silver_net or 0, 2) == round(gold_net or 0, 2), \
    f"net_amount differs: silver {silver_net} vs gold {gold_net}"
assert silver_sold == gold_sold, "units_sold differs between silver and gold"
for t, keys in [("fact_sales", ["date_key", "store_key", "product_key"]),
                ("fact_store_stock", ["date_key", "store_key", "product_key"]),
                ("fact_warehouse_stock", ["date_key", "product_key"]),
                ("fact_replenishment_decisions", ["date_key", "store_key", "product_key"])]:
    nulls = spark.table(f"{CATALOG}.{GOLD}.{t}").filter(
        " OR ".join(f"{k} IS NULL" for k in keys)).count()
    assert nulls == 0, f"{t} has {nulls} rows with NULL keys"
print("Integrity checks passed: row counts, net_amount, units_sold and no NULL keys.")
print("\nGold tables:")
for t in ["dim_date", "dim_store", "dim_product", "fact_sales", "fact_store_stock",
          "fact_warehouse_stock", "fact_replenishment_decisions"]:
    n = spark.table(f"{CATALOG}.{GOLD}.{t}").count()
    print(f"  {t:<30}{n:>10,} rows")

display(fs.orderBy("bill_date", "store_code").limit(5))

Integrity checks passed: row counts, net_amount, units_sold and no NULL keys.

Gold tables:
  dim_date                              90 rows
  dim_store                             40 rows
  dim_product                          440 rows
  fact_sales                       135,067 rows
  fact_store_stock                  11,199 rows
  fact_warehouse_stock              79,200 rows
  fact_replenishment_decisions      11,199 rows


date_key,store_key,product_key,bill_date,store_code,ean,units_sold,units_returned,gross_amount,discount_amount,net_amount,txn_line_count
20260718,1,78,2026-07-18,S1001,8900000000078,1,0,2399.0,0.0,2399.0,1
20260718,1,211,2026-07-18,S1001,8900000000211,1,0,1099.0,0.0,1099.0,1
20260718,1,168,2026-07-18,S1001,8900000000168,1,0,1299.0,0.0,1299.0,1
20260718,1,406,2026-07-18,S1001,8900000000406,1,0,4199.0,419.9,3779.1,1
20260718,1,226,2026-07-18,S1001,8900000000226,1,0,999.0,0.0,999.0,1
